# 0. Import Libraries

In [15]:
import os, sys

def add_project_path(module_folder="model_implementation"):
    candidates = [
        os.path.abspath("."),   # current directory (Deepnote case)
        os.path.abspath("../src/")   # parent directory (local notebooks case)
    ]

    for path in candidates:
        if os.path.exists(os.path.join(path, module_folder)):
            if path not in sys.path:
                sys.path.append(path)
            return path

    raise ImportError(f"Could not find '{module_folder}' in current or parent directory")

add_project_path("model_implementation")
add_project_path("cnn")

'c:\\Users\\Aryo\\PersonalMade\\ITB Kuliah Semesteran\\Semester 6\\IF3270 Pembelajaran Mesin\\Tubes\\ML-Tubes-2_RecursiveLearnaholic\\src'

In [16]:
import subprocess
import sys
print(sys.executable)

subprocess.check_call([sys.executable, "-m", "pip", "install", "tensorflow==2.21.0", "scikit-learn", "pandas", "matplotlib"])

import tensorflow as tf
print("TensorFlow successfully imported:", tf.__version__)

c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-Tubes-2_RecursiveLearnaholic\venv\Scripts\python.exe
TensorFlow successfully imported: 2.21.0


In [17]:
from pathlib import Path
from datetime import datetime

import joblib
import itertools
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

from model_implementation.ffnn import FFNN
from model_implementation.layer.loss import BCELoss
from cnn.layers import Conv2D, LocallyConnected2D, MaxPooling2D, AveragePooling2D, GlobalMaxPooling2D, GlobalAveragePooling2D, Flatten
from cnn.utility import image_loader, batch_loader, image_paths

# 1. Global Variables

In [19]:
SEED = 42
DEVICE = 'cpu'
IMAGE_SIZE = (150, 150) 
BATCH_SIZE = 32

plt.style.use("seaborn-v0_8")
np.random.seed(SEED)

CATEGORIES = {
    'buildings': 0, 
    'forest': 1,
    'glacier': 2,
    'mountain': 3,
    'sea': 4,
    'street': 5 
}
INV_CATEGORIES = {v: k for k, v in CATEGORIES.items()}

gpu_devices = tf.config.list_physical_devices('GPU')
if gpu_devices:
    print(f"GPU Detected: {gpu_devices}")
    for gpu in gpu_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    DEVICE = "/GPU:0" 
else:
    print("No GPU found, defaulting to CPU.")
    DEVICE = "/CPU:0"

No GPU found, defaulting to CPU.


# 2. Loading Data

In [ ]:
def load_intel_dataset(root_path, target_size=(150, 150)):
    """
    Loads images and labels from the directory structure:
    root/label/image.jpg
    """
    X = []
    y = []
    
    root = Path(root_path)
    for category, label in CATEGORIES.items():
        cat_path = root / category
        if not cat_path.exists():
            continue
            
        print(f"Loading {category}...")
        paths = image_paths(cat_path)
        
        for p in paths:
            try:
                img = image_loader(p, target_size=target_size)
                X.append(img)
                y.append(label)
            except Exception as e:
                print(f"Error loading {p}: {e}")
                
    return np.array(X, dtype="float32"), np.array(y, dtype="int32")

TRAIN_DIR = Path("../data/seg_train/seg_train")
TEST_DIR = Path("../data/seg_test/seg_test")
PRED_DIR = Path("../data/seg_pred/seg_pred")

print("--- Loading Training Data ---")
X_train, y_train = load_intel_dataset(TRAIN_DIR, target_size=IMAGE_SIZE)

print("\n--- Loading Test Data ---")
X_test, y_test = load_intel_dataset(TEST_DIR, target_size=IMAGE_SIZE)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=SEED)

print(f"\nFinal Shapes:")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

--- Loading Training Data ---
Loading buildings...
Loading forest...
Loading glacier...
Loading mountain...
Loading sea...
Loading street...


In [ ]:
# 1. Configuration variations for the 16 architectures [cite: 90]
keras_variations = {
    'num_layers': [2, 3],        
    'filters': [32, 64],       
    'kernel_size': [3, 5],     
    'pooling_type': ['max', 'avg']  
}

keras_keys, keras_values = zip(*keras_variations.items())
keras_experiments_configs = [dict(zip(keras_keys, v)) for v in itertools.product(*keras_values)]

keras_results_registry = []

for i, keras_cfg in enumerate(keras_experiments_configs):
    print(f"\n--- Running Keras Experiment {i+1}/16: {keras_cfg} ---")

    keras_model = models.Sequential()
    keras_model.add(layers.Conv2D(
        keras_cfg['filters'], 
        (keras_cfg['kernel_size'], keras_cfg['kernel_size']), 
        activation='relu', 
        padding='same',
        input_shape=(150, 150, 3)
    ))

    for _ in range(keras_cfg['num_layers'] - 1):
        keras_model.add(layers.Conv2D(
            keras_cfg['filters'], 
            (keras_cfg['kernel_size'], keras_cfg['kernel_size']), 
            activation='relu', 
            padding='same'
        ))

        if keras_cfg['pooling_type'] == 'max':
            keras_model.add(layers.MaxPooling2D((2, 2))) 
        else:
            keras_model.add(layers.AveragePooling2D((2, 2)))
            
    keras_model.add(layers.Flatten()) # [cite: 78]
    keras_model.add(layers.Dense(6, activation='softmax')) # 6 Classes for Intel Dataset 

    keras_model.compile(
        optimizer='adam', 
        loss='sparse_categorical_crossentropy', # [cite: 61]
        metrics=['accuracy']
    )

    keras_history = keras_model.fit(
        X_train, y_train, 
        epochs=10, 
        batch_size=32, 
        validation_data=(X_val, y_val), 
        verbose=1
    )

    keras_raw_preds = keras_model.predict(X_test, verbose=0)
    keras_y_pred = np.argmax(keras_raw_preds, axis=1)
    keras_f1_macro = f1_score(y_test, keras_y_pred, average='macro')

    keras_results_registry.append({
        'experiment_id': i,
        'config': keras_cfg, 
        'f1_macro': keras_f1_macro, 
        'history': keras_history.history,
        'weights': keras_model.get_weights() 
    })

# Final DataFrame for quick lookup in the notebook
df_keras_benchmarks = pd.DataFrame(keras_results_registry)
print("\n[Status] All 16 Keras experiments completed and stored in 'keras_results_registry'.")


--- Running Keras Experiment 1/16: {'num_layers': 2, 'filters': 32, 'kernel_size': 3, 'pooling_type': 'max'} ---


c:\Users\Aryo\PersonalMade\ITB Kuliah Semesteran\Semester 6\IF3270 Pembelajaran Mesin\Tubes\ML-Tubes-2_RecursiveLearnaholic\venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/10
351/351 ━━━━━━━━━━━━━━━━━━━━ 48s 132ms/step - accuracy: 0.6222 - loss: 1.0271 - val_accuracy: 0.7111 - val_loss: 0.8195
Epoch 2/10
268/351 ━━━━━━━━━━━━━━━━━━━━ 10s 122ms/step - accuracy: 0.7852 - loss: 0.6154

KeyboardInterrupt: 